# Incompressible flow

How does a viscous fluid flow around an obstacle? We use the steady Stokes equations, the linear model for slow incompressible flow.

## Velocity and pressure

The unknowns are the velocity $\vec u$ and pressure $p$:

$$
-\nu\Delta \vec u+\nabla p=f,
\qquad
\operatorname{div}\vec u=0
\qquad\text{in }\Omega.
$$

The first equation balances viscous forces, pressure, and external forces. The second expresses incompressibility: fluid volume is neither created nor destroyed. We prescribe an inflow profile, impose no slip, $\vec u=0$, on the walls and obstacle, and leave the outlet open with a natural outflow condition.

In [ ]:
from netgen.occ import OCCGeometry, Rectangle, X, Y
from ngsolve import (
    Mesh, H1, VectorH1, GridFunction, BilinearForm, CF,
    InnerProduct, Grad, div, dx, y, Norm
)
from ngsolve.webgui import Draw

channel = Rectangle(2, 0.41).Circle(0.2, 0.2, 0.05).Reverse().Face()
channel.edges.name = "obstacle"
channel.edges.Min(X).name = "inlet"
channel.edges.Max(X).name = "outlet"
channel.edges.Min(Y).name = "wall"
channel.edges.Max(Y).name = "wall"
mesh = Mesh(OCCGeometry(channel, dim=2).GenerateMesh(maxh=0.09)).Curve(2)
Draw(mesh);

In [ ]:
# Numerical solver - mixed finite elements will be introduced later
velocity_space = VectorH1(mesh, order=2, dirichlet="inlet|wall|obstacle")
pressure_space = H1(mesh, order=1)
space = velocity_space * pressure_space
(u, p), (v, q) = space.TnT()

viscosity = 0.001
A = BilinearForm(space)
A += (viscosity*InnerProduct(Grad(u), Grad(v)) - div(u)*q - div(v)*p - 1e-8*p*q) * dx
A.Assemble()

solution = GridFunction(space)
velocity, pressure = solution.components

inflow = CF((1.5*4*y*(0.41-y)/(0.41**2), 0))
velocity.Set(inflow, definedon=mesh.Boundaries("inlet"))
Draw(velocity, mesh, "inflow profile")

residual = -A.mat * solution.vec
solution.vec.data += A.mat.Inverse(space.FreeDofs(), inverse="sparsecholesky") * residual

In [ ]:
Draw(Norm(velocity), mesh, "speed")
Draw(velocity, mesh, "velocity", vectors={"grid_size": 32})
Draw(pressure, mesh, "pressure");

## Observe

- Where does the flow accelerate around the obstacle?
- Where are the velocity vectors zero?
- Why do velocity and pressure have to be solved for together?


[← Linear elasticity](04_linear_elasticity.ipynb) · [Lecture overview](index.ipynb) · [Next: magneto-quasistatics →](06_magneto_quasistatics.ipynb)